# Unit tests generator

We will use frontier model to generate unit tests of a python module

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print(f"OpenAI API Key exists and begins {api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
base_url = "https://openrouter.ai/api/v1"
openai = OpenAI(base_url=base_url, api_key=api_key)

In [ ]:
anthropic = OpenAI(api_key=api_key, base_url=base_url)
gemini = OpenAI(api_key=api_key, base_url=base_url)
grok = OpenAI(api_key=api_key, base_url=base_url)
groq = OpenAI(api_key=api_key, base_url=base_url)
ollama = OpenAI(api_key="ollama", base_url="http://127.0.0.1:11434/v1")
openrouter = OpenAI(api_key=api_key, base_url=base_url)

In [ ]:
models = ["gpt-5-nano", "claude-sonnet-4-5-20250929", "grok-4", "gemini-2.5-pro", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b", ]

clients = {"gpt-5-nano": openai, "claude-sonnet-4-5-20250929": anthropic, "grok-4": grok, "gemini-2.5-pro": gemini, "openai/gpt-oss-120b": groq, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama, "qwen/qwen3-coder-30b-a3b-instruct": openrouter}

In [ ]:
system_prompt = """
You are a senior Python test engineer.

Generate production-ready unit tests for the provided Python module using pytest.

Requirements:
- Cover all public functions and class methods.
- Include happy paths, edge cases, boundary values, invalid inputs, and exception cases.
- Mock all external dependencies (API calls, DB, filesystem, env vars, time).
- Use pytest fixtures and parametrization when appropriate.
- Use pytest.raises for exceptions.
- Keep tests deterministic (no randomness or real external calls).
- Use clear test names: test_<function>_<scenario>.
- Return ONLY the complete runnable test file.
- No explanations. No markdown. Code only.
"""

def user_prompt_for(python):
    return f"""
    Generate unit tests for the following Python code:

    ```python
    {python}
    ```
    """

In [ ]:
# Format LLM messages
def format_messages(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

In [ ]:
def generate_unit_tests(model, python_code):
    try:
        client = clients[model]
        messages = format_messages(python_code)
        
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0,
            tool_choice="none",
        )

        content = response.choices[0].message.content

        if not content:
            return "Error: Model returned empty response."

        # Create test filename
        test_filename = "test_generated.py"

        # Write tests to file
        with open(test_filename, "w", encoding="utf-8") as f:
            f.write(content)

        return content

    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
with gr.Blocks() as demo:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28)
        python_test = gr.Textbox(label="Unit tests:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Generate tests")

    convert.click(generate_unit_tests, inputs=[model, python], outputs=[python_test])

demo.launch(inbrowser=True)